# Coffee17 multistage recalibration — fail-fast
Screening validation-only seed 42 untuk MSF0 (fixed multistage) dan MSF1 (adaptive stage-channel recalibration), dibandingkan dengan EfficientNetV2-B0 GAP/HBP. Test tetap terkunci. Checkpoint disimpan ke Drive dan Hugging Face setiap epoch agar dapat dilanjutkan lintas akun Colab.

In [ ]:
SEEDS = [42]
REPO_REF = 'agent/sni-instance-crops'
HF_REPO = 'ediprin/coffee-backbone-checkpoints'
HF_NAMESPACE = 'coffee17-multistage-recalibration-v1'
HF_SYNC_EVERY = 1
DRIVE_FOLDER = 'coffee17-multistage-recalibration-v1'
EVALUATION_SPLIT = 'val'

In [ ]:
# 1/6 — SETUP REPO, DRIVE, GPU, HF, DAN COFFEE17
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
assert SEEDS == [42] and EVALUATION_SPLIT == 'val'
repo = Path('/content/coffee-bean-classification')
repo_url = 'https://github.com/ediprin/coffee-bean-classification.git'
def run(command, cwd=None):
    command = [str(item) for item in command]
    print('\n$', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)
if (repo / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_REF], cwd=repo)
    run(['git', 'checkout', REPO_REF], cwd=repo)
    run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=repo)
else:
    if repo.exists(): shutil.rmtree(repo)
    run(['git', 'clone', '--branch', REPO_REF, repo_url, repo])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo])
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
token = userdata.get('HF_TOKEN')
assert token, 'Tambahkan secret HF_TOKEN dengan izin write di Colab.'
os.environ['HF_TOKEN'] = token
os.environ['BILINEAR_LMMD_ARTIFACT_REPO'] = HF_REPO
os.environ['BILINEAR_LMMD_ARTIFACT_NAMESPACE'] = HF_NAMESPACE
os.environ['BILINEAR_LMMD_ARTIFACT_SYNC_EVERY'] = str(HF_SYNC_EVERY)
os.environ['BILINEAR_LMMD_ARTIFACT_REQUIRED'] = '1'
from huggingface_hub import HfApi, login
login(token=token, add_to_git_credential=False)
hf_user = HfApi().whoami(token=token)['name']
dataset_base = Path('/content/coffee17-msf-data')
original, clean = dataset_base / 'original', dataset_base / 'clean'
archive = dataset_base / 'coffee17.zip'
data_root = clean / 'folds/fold_1'
suffixes = {'.jpg', '.jpeg', '.png'}
def image_count(path):
    return sum(1 for p in path.rglob('*') if p.is_file() and p.suffix.lower() in suffixes) if path.is_dir() else 0
if image_count(original / 'source') != 979:
    if original.exists(): shutil.rmtree(original)
    if archive.exists() and not zipfile.is_zipfile(archive): archive.unlink()
    run([sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_coffee17', '--output', original, '--archive', archive, '--seed', '42'], cwd=repo)
if not (clean / 'audit.json').is_file():
    if clean.exists(): shutil.rmtree(clean)
    run([sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_clean_grouped_folds', '--source-root', original / 'source', '--output-root', clean, '--expected-count', '979', '--folds', '5', '--seed', '42', '--validation-ratio', '0.1'], cwd=repo)
assert all((data_root / f'source/{split}').is_dir() for split in ('train', 'val', 'test'))
output_root = Path('/content/drive/MyDrive') / DRIVE_FOLDER
output_root.mkdir(parents=True, exist_ok=True)
baseline_root = Path('/content/backbone-results')
print('GPU       :', torch.cuda.get_device_name(0))
print('HF        :', hf_user, '->', HF_REPO, '/', HF_NAMESPACE)
print('DATASET   :', data_root)
print('OUTPUT    :', output_root)
print('CHECKPOINT: setiap', HF_SYNC_EVERY, 'epoch | TEST LOCKED')

In [ ]:
# 2/6 — PULIHKAN BASELINE BE2G/BE2H
from huggingface_hub import snapshot_download
patterns = []
for code in ('BE2G', 'BE2H'):
    patterns += [f'outputs/{code}_seed42/*', f'val_reports/{code}_seed42/*']
snapshot_download(repo_id=HF_REPO, repo_type='model', token=token, local_dir=baseline_root, allow_patterns=patterns)
command = [sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_backbone_screening', '--data-root', str(data_root), '--output-root', str(baseline_root), '--seeds', '42', '--backbones', 'EV2', '--heads', 'gap', 'hbp', '--evaluation-split', 'val', '--hf-repo', HF_REPO, '--hf-sync-every', '1']
print('MENYIAPKAN BASELINE:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=repo)
while process.poll() is None:
    print('[BASELINE] restore/evaluasi masih berjalan', flush=True)
    time.sleep(60)
assert process.wait() == 0, 'Baseline gagal; baca traceback.'
print('BASELINE SIAP')

In [ ]:
# 3/6 — AUDIT ARSITEKTUR TANPA TRAINING
audit_path = output_root / 'architecture_audit.json'
run([sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.audit_multistage_recalibration', '--output', audit_path, '--models', 'BE2G', 'BE2H', 'MSF0', 'MSF1', 'MSFC', '--device', 'cuda', '--warmup', '5', '--repeats', '20'], cwd=repo)

In [ ]:
# 4/6 — TRAIN / RESUME MSF0 DAN MSF1
command = [sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_multistage_recalibration_screening', '--data-root', str(data_root), '--baseline-root', str(baseline_root), '--output-root', str(output_root), '--seeds', '42', '--evaluation-split', EVALUATION_SPLIT, '--hf-repo', HF_REPO, '--hf-namespace', HF_NAMESPACE, '--hf-sync-every', str(HF_SYNC_EVERY)]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=repo)
started = time.monotonic()
while process.poll() is None:
    rows = []
    for code in ('MSF0', 'MSF1'):
        history = output_root / f'outputs/{code}_seed42/history.json'
        if history.is_file():
            try: rows.append(f'{code}={len(json.loads(history.read_text()))}/50')
            except Exception: rows.append(f'{code}=saving')
    print(f'[MSF {(time.monotonic()-started)/60:.1f} menit]', ', '.join(rows) if rows else 'inisialisasi', flush=True)
    time.sleep(60)
assert process.wait() == 0, 'Screening gagal; baca traceback.'

In [ ]:
# 5/6 — TAMPILKAN HASIL SCREENING
path = output_root / 'val_reports/multistage_recalibration_screening.json'
assert path.is_file(), f'Report belum ditemukan: {path}'
report = json.loads(path.read_text())
for name, summary in report['comparisons'].items():
    print(f'\n=== {name} ===')
    for key in ('macro_f1', 'hard_class_f1', 'bottom3_class_f1', 'worst_class_f1'):
        row = summary[key]
        print(f"{key:22s}: {row['baseline_mean']:.2%} -> {row['candidate_mean']:.2%} Delta={row['delta_mean']:+.2%}")
print('\n=== PUTUSAN ===')
for name, row in report['decisions'].items(): print(name, row['decision'], row['criteria'])
print('FINAL:', report['final_decision'])
print('Test dibuka:', report['test_opened'])

In [ ]:
# 6/6 — CAPACITY-MATCHED CONTROL: JALANKAN HANYA SETELAH SCREEN PASS
# Runner memulihkan report screening dari HF dan memverifikasi PASS sebelum training.
command = [sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_multistage_recalibration_screening', '--data-root', str(data_root), '--baseline-root', str(baseline_root), '--output-root', str(output_root), '--seeds', '42', '--evaluation-split', EVALUATION_SPLIT, '--stage', 'capacity', '--hf-repo', HF_REPO, '--hf-namespace', HF_NAMESPACE, '--hf-sync-every', str(HF_SYNC_EVERY)]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=repo)
started = time.monotonic()
while process.poll() is None:
    history = output_root / 'outputs/MSFC_seed42/history.json'
    status = 'inisialisasi'
    if history.is_file():
        try: status = f"MSFC={len(json.loads(history.read_text()))}/50"
        except Exception: status = 'MSFC=saving'
    print(f'[MSFC {(time.monotonic()-started)/60:.1f} menit] {status}', flush=True)
    time.sleep(60)
assert process.wait() == 0, 'Capacity control gagal; baca traceback.'
capacity_path = output_root / 'val_reports/multistage_recalibration_capacity_control.json'
capacity = json.loads(capacity_path.read_text())
for name, summary in capacity['comparisons'].items():
    print(f'\n=== {name} ===')
    for key in ('macro_f1', 'hard_class_f1', 'bottom3_class_f1', 'worst_class_f1'):
        row = summary[key]
        print(f"{key:22s}: {row['baseline_mean']:.2%} -> {row['candidate_mean']:.2%} Delta={row['delta_mean']:+.2%}")
print('\n=== PUTUSAN CAPACITY CONTROL ===')
for name, row in capacity['decisions'].items(): print(name, row['decision'], row['criteria'])
print('FINAL:', capacity['final_decision'], '| Test dibuka:', capacity['test_opened'])

In [ ]:
# 7/7 — AUDIT CAPACITY TANPA TRAINING DAN TANPA MEMBUKA TEST
# Jalankan setelah capacity control selesai, termasuk ketika FINAL: FAIL.
command = [sys.executable, '-u', '-m', 'bilinear_lmmd.analysis.multistage_capacity', '--baseline-root', str(baseline_root), '--output-root', str(output_root), '--seed', '42', '--iterations', '10000']
print('MENJALANKAN AUDIT:', ' '.join(command), flush=True)
run(command, cwd=repo)
audit_path = output_root / 'val_reports/capacity_audit_seed42/multistage_capacity_audit.json'
audit = json.loads(audit_path.read_text())
print('\n=== KELAS PALING TURUN/NAIK: MSFC vs MSF1 ===')
import pandas as pd
per_class = pd.read_csv(output_root / 'val_reports/capacity_audit_seed42/per_class_deltas.csv')
display(per_class[per_class.comparison == 'MSFC_vs_MSF1'].sort_values('delta_f1')[['class', 'support', 'baseline_f1', 'candidate_f1', 'delta_f1', 'rescued_by_candidate', 'harmed_by_candidate']])
print('\nAudit diagnostik selesai. Seed 123/2026 dan test tetap terkunci.')

## Aturan berhenti
Jika screening `FINAL: FAIL`, MSF dihentikan. Jika screening PASS, jalankan sel capacity control. Jika capacity control FAIL, jangan membuka seed 123/2026 atau test; jalankan hanya audit post-hoc tanpa training untuk menjelaskan kegagalan. Bila runtime reset atau akun Colab diganti, tambahkan `HF_TOKEN` yang sama lalu jalankan ulang sel setup dan capacity; runner memulihkan epoch terakhir dari Hugging Face.